# Why the LLM's Answer Keeps Failing Validation

In this video, I'm going to ask Gemini about the same five photos two ways, with one Pydantic model. With the schema only in the prompt, all five answers failed validation. With the schema in the request, all five passed.

When an LLM's answer about your image fails validation, the error alone doesn't tell you why. In Pixeltable, a multimodal backend, each answer goes in its own column, and its check goes in the next column, on the same row as its photo.

This question comes from a Pydantic AI issue called "Structured output validation keeps failing." First we'll find out why our answers failed, and then what the check covers and what it doesn't.

## Start from an empty `llm_output_demo` directory

Start Jupyter with `uv run jupyter lab` in this folder, with your Gemini key in `GEMINI_API_KEY`. Each photo gets two paid calls to Gemini, and we'll use six photos, so that's 12 calls.

In Pixeltable, a directory is a namespace for tables. We'll drop ours first, so every run starts empty.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt
import pixeltable.functions as pxtf

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('llm_output_demo', force=True, if_not_exists='ignore')

## `ListingDetails` is the answer we want for each photo

`ListingDetails` is the Pydantic model from a marketplace app we built for our Pydantic webinar. `MODEL` is the Gemini model both calls use.

`category` and `condition` are `Literal` types, so each one allows only the values listed.

In [ ]:
!sed -n '8,19p' listing.py

## The schema goes in the prompt, or in the request

The first call pastes the JSON schema of `ListingDetails` into the prompt text. Here's how that prompt is built, and the whole prompt Gemini gets, next to the photo.

In [ ]:
!sed -n '22,25p' listing.py
from listing import PROMPT_WITH_SCHEMA

print(PROMPT_WITH_SCHEMA)

The second call sends only `PROMPT`, the first line above. The schema goes in the request: `json_config` sets it as Gemini's `response_json_schema`.

In [ ]:
!sed -n '28,30p' listing.py

## Each answer and its check get their own column

In Pixeltable, you declare tables as Python classes. In `photos`, the only columns you fill in are `name` and `photo`. The rest are computed columns, which run on every insert.

In [ ]:
!sed -n '6,22p' schema.py

`prompt_only` holds Gemini's whole answer to the first call, and `prompt_only_text` is its text. `prompt_only_details` parses that text with `json.loads`, then `astype(pxt.Json[ListingDetails])` checks its keys and types.

`response` holds the answer to the second call, and `details` runs the same check on its `parsed` field. Gemini's SDK fills in `parsed` only when the request has a schema, which is why the first check parses the text.

`pxt schema diff` shows what's about to change. Then `pxt schema update` creates the table.

In [ ]:
!pxt schema diff schema.py llm_output_demo

In [ ]:
!pxt schema update schema.py llm_output_demo -f

## A failed check keeps the answer on its photo's row

I'll insert five photos from the marketplace app. Each `photo` is a URL, and the insert downloads the photo, so Gemini gets the image.

`on_error='ignore'` keeps a row whose check fails, along with its error. Without it, one failed check stops the whole insert, and no rows are saved.

In [ ]:
BASE = 'https://raw.githubusercontent.com/apreshill/pixeltable-marketplace/main/samples/'
names = ['film-camera-1', 'cruiser-bike-1', 'guitar-1', 'backpack-1', 'salt-lamp-1']

photos = pxt.get_table('llm_output_demo.photos')
photos.insert([{'name': name, 'photo': f'{BASE}{name}.jpg'} for name in names], on_error='ignore')

That's ten calls to Gemini, and five errors, all in `prompt_only_details`.

## With the schema only in the prompt, every answer failed `json.loads`

`errortype` and `errormsg` hold each failure's exception, on the row of its photo.

In [ ]:
photos.where(photos.prompt_only_details.errortype != None).select(
    photos.name, photos.photo, photos.prompt_only_details.errortype, photos.prompt_only_details.errormsg
).collect()

All five fail at line 1, column 1, where `json.loads` expected the JSON to start. The error says where, and the raw answer says why.

In [ ]:
for row in photos.select(photos.name, photos.prompt_only_text).collect():
    print(f"--- {row['name']}\n{row['prompt_only_text']}")

Gemini wrapped every answer in a Markdown code fence, so the text starts with three backticks, not a brace. In this run, the JSON inside each fence would pass the check.

## With the schema in the request, every answer passed

These are the same photos and the same Pydantic model. The only change is where the schema went. Here's the text of the guitar's answer this time.

In [ ]:
guitar = photos.where(photos.name == 'guitar-1')
print(guitar.select(text=photos.response.candidates[0].content.parts[0].text).collect()[0]['text'])

It starts with a brace, with no fence. Here are all five.

In [ ]:
photos.select(
    photos.name, photos.photo, photos.details.category, photos.details.condition, photos.details.errortype
).collect()

No errors, and every `category` and `condition` is one of the allowed values.

## `pxt.Json[ListingDetails]` checks keys and types, not `Literal` values

`pxt.Json[ListingDetails]` checks that every required key is there, with the right type. It doesn't check that a `Literal` field holds one of its values.

Gemini's answers kept to the allowed values, so to show one that doesn't, I'll use a stand-in. This is the demo-only part: it passes the answer from `response` through, except on `desk-lamp-1`, where it changes `condition` to `'barely used'`.

In [ ]:
!sed -n '14,23p' udfs.py

## A UDF with `model_validate` catches the `Literal` value

A UDF is a Python function that a computed column can call. For Pydantic's full validation, `validate_listing` calls `ListingDetails.model_validate`.

In [ ]:
!sed -n '8,11p' udfs.py

`schema_validate.py` adds three columns: the stand-in's answer, and that answer checked both ways. The diff marks all three as safe.

In [ ]:
!diff schema.py schema_validate.py

In [ ]:
!pxt schema diff schema_validate.py llm_output_demo

In [ ]:
!pxt schema update schema_validate.py llm_output_demo -f

The update computed the three columns for the five photos from the answers already stored. It didn't call Gemini again.

Now I'll insert the desk lamp.

In [ ]:
photos = pxt.get_table('llm_output_demo.photos')  # reload after the schema change
photos.insert([{'name': 'desk-lamp-1', 'photo': f'{BASE}desk-lamp-1.jpg'}], on_error='ignore')

Two errors this time: `prompt_only_details` again, and `stand_in_validated`.

In [ ]:
photos.select(
    photos.name,
    condition=photos.stand_in_details.condition,
    details_error=photos.stand_in_details.errortype,
    validated_error=photos.stand_in_validated.errortype,
).collect()

`'barely used'` got through `stand_in_details` with no error. `stand_in_validated` caught it, and its error is Pydantic's own message.

In [ ]:
for row in photos.where(photos.stand_in_validated.errortype != None).select(
    photos.name, photos.stand_in_validated.errormsg
).collect():
    print(f"--- {row['name']}\n{row['stand_in_validated_errormsg']}")

In your own table, when your code depends on a `Literal` value, check `response.parsed` with `validate_listing` instead of `astype`.

## Each failed answer sat next to its error

We asked Gemini about the same photos two ways. With the schema only in the prompt, every answer came back in a Markdown fence and failed `json.loads`. With the schema in the request, every answer passed.

Then the `astype` check let a `Literal` value through, and a UDF with `model_validate` caught it.

Pixeltable doesn't make an LLM return valid output. What you get is each answer in a column next to its check, on its photo's row, so you can see what came back and why it failed.

The notebook and schema files are in the `check-llm-output` folder of the repo linked below. If this was useful, please consider starring the Pixeltable GitHub repo. Stars help other developers find it.